In [94]:
from geneticengine.grammar.decorators import abstract
from geneticengine.grammar.grammar import extract_grammar
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.random.sources import NativeRandomSource
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.problems import SingleObjectiveProblem
from geneticengine.evaluation.budget import EvaluationBudget, TimeBudget
from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.problems import MultiObjectiveProblem
from geneticengine.grammar.decorators import abstract
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection
from geneticengine.algorithms.gp.operators.combinators import SequenceStep, ParallelStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import TournamentSelection


from typing import Annotated
from dataclasses import dataclass
from abc import ABC

import numpy as np
from numpy.random import choice
import pandas as pd
import re
import time

from sklearn import tree
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar.metahandlers.lists import ListSizeBetween

In [95]:
df = pd.read_csv('../../datasets/synthetic_classification_data.csv')
X = df.drop(columns=['target'])
y = df['target']
n_features = X.shape[1]
feature_names = X.columns

In [96]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [97]:
importances = rf.feature_importances_
sorted_indices = np.argsort(importances)[::-1]
split_point = int(n_features*0.2)

In [98]:
exploit_indices = sorted_indices[:split_point]
exploit_importances = importances[exploit_indices]
exploit_probabilities = exploit_importances / np.sum(exploit_importances)

In [99]:
explore_indices = sorted_indices[split_point:]
explore_importances = importances[explore_indices]
explore_probabilities = explore_importances / np.sum(explore_importances)

In [100]:
print(f"Total Features: {n_features}")
print(f"Exploitation Swarn will use {len(exploit_indices)} features")
print(f"Exploration Swarn will use {len(explore_indices)} features")

Total Features: 10
Exploitation Swarn will use 2 features
Exploration Swarn will use 8 features


In [101]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train = X_train.values
X_test = X_test.values

In [102]:
@abstract
class Feature(ABC):
    def evaluate(self, X):
        pass

@abstract
class BasePrimitive(Feature):
    index: int
    def evaluate(self, X):
        return X[:, self.index]
    def __str__(self):
        name = feature_names[self.index]
        safe = re.sub(r'[^A-Za-z0-9_]+', '_', str(name))
        return f"f_{safe}"

@dataclass
class ExploiterPrimitive(BasePrimitive):
    def __init__(self):
        self.index = choice(exploit_indices, p=exploit_probabilities)
    
@dataclass
class ExplorerPrimitive(BasePrimitive):
    def __init__(self):
        self.index = choice(explore_indices, p=explore_probabilities)

@dataclass
class Sum(Feature):
    terms: Annotated[list[Feature], ListSizeBetween(2, min(5, n_features))]
    
    def evaluate(self, X):
        results = [term.evaluate(X) for term in self.terms]
        valid_results = [r for r in results if r is not None]
        if not valid_results:
            return np.zeros(X.shape[0])
        return np.sum(valid_results, axis=0)

    def __str__(self):
        return "sum(" + ", ".join(str(term) for term in self.terms) + ")"

@dataclass
class Multiplication(Feature):
    terms: Annotated[list[Feature], ListSizeBetween(2, min(5, n_features))]
    
    def evaluate(self, X):
        results = [term.evaluate(X) for term in self.terms]
        valid_results = [r for r in results if r is not None]
        if not valid_results:
            return np.ones(X.shape[0])
        return np.prod(valid_results, axis=0)

    def __str__(self):
        return "mul(" + ", ".join(str(term) for term in self.terms) + ")"

@dataclass
class FeatureSet:
    features: Annotated[list[Feature], ListSizeBetween(3,8)]
    def evaluate_all(self, X):
        evaluated_features = []
        for feature in self.features:
            try:
                result = feature.evaluate(X)
                if result is not None and isinstance(result, np.ndarray) and result.ndim == 1 and len(result) == X.shape[0]:
                    evaluated_features.append(result)
                else:
                    evaluated_features.append(np.zeros(X.shape[0]))
            except Exception:
                evaluated_features.append(np.zeros(X.shape[0]))
        if not evaluated_features:
            return np.empty((X.shape[0], 0))
        return np.column_stack(evaluated_features)

    def get_feature_count(self):
        return len(self.features)
    def __str__(self):
        return "FeatureSet(" + ", ".join(str(f) for f in self.features) + ")"


exploit_components = [
    Feature,
    BasePrimitive,
    ExploiterPrimitive,
    Sum, Multiplication,
    FeatureSet
]

exploit_grammar = extract_grammar(exploit_components, FeatureSet)
print(exploit_grammar)

explorer_components = [
    Feature,
    BasePrimitive,
    ExplorerPrimitive,
    Sum, Multiplication,
    FeatureSet
]

explorer_grammar = extract_grammar(explorer_components, FeatureSet)
print(explorer_grammar)

Grammar<Starting=FeatureSet,Productions={
Feature -> BasePrimitive()|
	Sum(terms: Annotated[list])|
	Multiplication(terms: Annotated[list])

BasePrimitive -> ExploiterPrimitive()
}
Grammar<Starting=FeatureSet,Productions={
Feature -> BasePrimitive()|
	Sum(terms: Annotated[list])|
	Multiplication(terms: Annotated[list])

BasePrimitive -> ExplorerPrimitive()
}


In [103]:
baseline_model = tree.DecisionTreeClassifier(max_depth=5, random_state=42)
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
baseline_scores = cross_val_score(baseline_model, X_train, y_train, cv=cv_strategy, scoring='accuracy')
print(f"Baseline CV Accuracy on original features: {np.mean(baseline_scores)}")

Baseline CV Accuracy on original features: 0.9183333333333333


In [104]:
def fitness_function(fs: FeatureSet) -> list[int]:
    model = tree.DecisionTreeClassifier(max_depth=5, random_state=42)

    X_transformed = fs.evaluate_all(X_train)
    
    model.fit(X_transformed, y_train)
    predictions = model.predict(X_transformed)
    error_vector = (predictions != y_train)
    return list(int(e) for e in error_vector)

In [105]:
rnd = NativeRandomSource(123)
decider = MaxDepthDecider(rnd, exploit_grammar, max_depth=5)
representation = TreeBasedRepresentation(exploit_grammar, decider)
problem = MultiObjectiveProblem(fitness_function=fitness_function, minimize=[False, True])

def lexicase_step():
    return ParallelStep(
        [
            ElitismStep(),
            NoveltyStep(),
            SequenceStep(
                LexicaseSelection(True),
                # TournamentSelection(5),
                GenericCrossoverStep(0.01),
                GenericMutationStep(0.9),
            ),
        ],
        weights=[5,5,90]
    )

gp = GeneticProgramming(
    problem=problem,
    budget=TimeBudget(30), 
    representation=representation,
    random=rnd,
    population_size=50,
    step=lexicase_step()
    
)

In [106]:
solutions = gp.search()

c:\Users\Gabriel\Desktop\Tese\.venv\Lib\site-packages\sklearn\utils\_array_api.py:839: RuntimeWarning: overflow encountered in cast
  array = numpy.asarray(array, order=order, dtype=dtype)
c:\Users\Gabriel\Desktop\Tese\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


ValueError: Input X contains infinity or a value too large for dtype('float32').

In [ ]:
best_solution = solutions[0]

print(best_solution.get_phenotype())
print(f"Fitness (CV Accuracy on training set): {best_solution.get_fitness(problem)}")

FeatureSet(mul(sum(mul(f_f3, f_f2, f_f3, f_f2), f_f3, mul(f_f3, f_f3, f_f2, f_f3)), sum(sum(f_f2, f_f3, f_f3, f_f3, f_f2), sum(f_f3, f_f3, f_f2)), sum(mul(f_f3, f_f2, f_f3, f_f3), sum(f_f3, f_f3, f_f3, f_f2, f_f2), sum(f_f3, f_f2, f_f3, f_f3, f_f3), sum(f_f3, f_f3, f_f3, f_f3))), sum(sum(sum(f_f3, f_f3, f_f2), sum(f_f3, f_f3, f_f3, f_f3, f_f3), mul(f_f3, f_f3)), mul(f_f3, mul(f_f2, f_f2, f_f3, f_f2, f_f3))), mul(mul(mul(f_f3, f_f3, f_f2), f_f3, f_f2), mul(f_f3, mul(f_f3, f_f3, f_f2, f_f3), f_f2, f_f3), f_f2, f_f2, sum(mul(f_f3, f_f3, f_f3, f_f3, f_f3), sum(f_f3, f_f3, f_f3, f_f2))), f_f2, sum(sum(f_f3, sum(f_f3, f_f3, f_f3, f_f3), mul(f_f3, f_f2, f_f3, f_f3), mul(f_f3, f_f3)), f_f3), sum(mul(mul(f_f2, f_f3, f_f3, f_f3, f_f3), f_f3), sum(mul(f_f3, f_f2, f_f2, f_f3), f_f3, sum(f_f3, f_f2, f_f2, f_f3, f_f3), sum(f_f3, f_f3, f_f3, f_f3, f_f2)), f_f3, sum(sum(f_f3, f_f2, f_f3, f_f3), sum(f_f3, f_f2, f_f3, f_f3, f_f3), mul(f_f3, f_f3, f_f3, f_f3, f_f3)), f_f3), f_f2)
Fitness (CV Accuracy on 